This file allows to analyze results obtained by running experiments_competing_risk SYNTHETIC_COMPETING_GROUP.

In [ ]:
import os 
import numpy as np
import scipy.stats
import pandas as pd
import matplotlib.pyplot as plt

from generate import *

import pickle
import sys
sys.path.append('../NeuralFineGray/')
sys.path.append('../NeuralFineGray/DeepSurvivalMachines/')

import seaborn as sns
custom_params = {"axes.spines.right": False, "axes.spines.top": False, "axes.spines.left": False,
                 "axes.spines.bottom": False, "figure.dpi": 700, 'savefig.dpi': 300}
sns.set_theme(style = "whitegrid", rc = custom_params, font_scale = 1.75)

In [ ]:
path = '../Results/' # Path where the data is saved
CAUSE = 2
DIM = 10

# Evaluate all models

In [ ]:
# Utils to compute the metrics 
from metrics import truncated_concordance_td, brier_score

def mse(pred, gt):
    return np.nan_to_num(np.sqrt(((pred - gt) ** 2).mean(1)), 0).mean()

### Utils: The evaluatino metrics used
def evaluate(survival_pred, survival_gt, e, t, times, times_eval, groups = None):
    results = {}
    train_index = survival_gt.index.difference(survival_pred.index)

    # If multiple risk, compute cause specific metrics
    for r in survival_pred.columns.get_level_values(0).unique():
        res = {}
        e_train, t_train = e[train_index].values, t[train_index].values
        e_test,  t_test  = e[survival_pred.index].values, t[survival_pred.index].values
        g_train, g_test = (None, None) if groups is None else (groups[train_index], groups[survival_pred.index])
        
        survival_fold = survival_pred[r]
        survival_gt_fold = survival_gt.loc[survival_pred.index][r]

        if len(times_eval) > 0:
            indexes = [np.argmin(np.abs(times - te)) for te in times_eval]
            km = (e_train, t_train)
            for index, horizon, te in zip(indexes, horizons, times_eval):
                ci, km = truncated_concordance_td(e_test, t_test, 1 - survival_fold.values, times, te, km = km, competing_risk = int(r))
                try:
                    res[horizon] = {
                        "SE": mse(1 - survival_fold.iloc[:, :index + 1], 1 - survival_gt_fold.iloc[:, :index + 1]),
                        "CIS": ci, 
                        "BRS": brier_score(e_test, t_test, 1 - survival_fold.values, times, te, km = km, competing_risk = int(r))[0]}
                except:
                    raise
            
                for group in groups.unique() if groups is not None else []:
                    try:
                        km = (e_train[g_train == group], t_train[g_train == group])
                        res[horizon]["CIS_{}".format(group)] = truncated_concordance_td(e_test[g_test == group], t_test[g_test == group], 1 - survival_fold[g_test == group].values, times, te, km = km, competing_risk = int(r))[0]
                        res[horizon]["BRS_{}".format(group)] = brier_score(e_test[g_test == group], t_test[g_test == group], 1 - survival_fold[g_test == group].values, times, te, km = km, competing_risk = int(r))[0]
                        res[horizon]["SE_{}".format(group)] = mse(1 - survival_fold[g_test == group].iloc[:, :index + 1], 1 - survival_gt_fold[g_test == group].iloc[:, :index + 1])

                        km = (e_train[g_train != group], t_train[g_train != group])
                        res[horizon]["Delta_CIS_{}".format(group)] = res[horizon]["CIS_{}".format(group)] - truncated_concordance_td(e_test[g_test != group], t_test[g_test != group], 1 - survival_fold[g_test != group].values, times, te, km = km, competing_risk = int(r))[0]
                        res[horizon]["Delta_BRS_{}".format(group)] = res[horizon]["BRS_{}".format(group)] - brier_score(e_test[g_test != group], t_test[g_test != group], 1 - survival_fold[g_test != group].values, times, te, km = km, competing_risk = int(r))[0]
                        res[horizon]["Delta_SE_{}".format(group)] = res[horizon]["SE_{}".format(group)] - mse(1 - survival_fold[g_test != group].iloc[:, :index + 1], 1 - survival_gt_fold[g_test != group].iloc[:, :index + 1])
                    
                    except:
                        pass
        results[r] = pd.DataFrame.from_dict(res)
    results = pd.concat(results)
    results.index.set_names(['Risk', 'Metric'], inplace = True)

    return results

In [ ]:
# Rename
dict_name = {'nfg': 'NeuralFG', 'nfgnc': 'NeuralFG Non Competing', 
             'finegray': 'Fine-Gray', 'cox': 'Cox',
             'dh': 'DeepHit', 'dhnc': 'DeepHit Non Competing',
             'ds': 'DeSurv', 'dsnc': 'DeSurv Non Competing', 
            } 

In [ ]:
predictions, results, theoretical = {}, {}, {}
horizons = [0.5, 1] # Horizons to evaluate the models

In [ ]:
# Helpers for the model-comparison loop below

def parse_seed_and_model(file_name):
    """Extract (random_seed, model_code) from a 'generate=<seed>_<model>.csv' filename."""
    random_seed = int(file_name[file_name.rindex('=') + 1: file_name.index('_')])
    model_code = file_name[file_name.rindex('_') + 1: file_name.index('.')]
    return random_seed, model_code


def load_predictions(path, file_name):
    """Load one model's prediction CSV into a DataFrame with a (cause, time) MultiIndex."""
    if 'finegray' in file_name or 'cox' in file_name:
        # Written without a MultiIndex header; reconstruct one assuming the columns
        # run [risk 1 x 100 times][risk 2 x 100 times][Use]
        preds = pd.read_csv(path + file_name, header = [0], index_col = 0).T.ffill().T
        index = pd.DataFrame([[i, t] for i in ('1', '2') for t in preds.columns[:100]] + [['Use', '']])
        preds.columns = pd.MultiIndex.from_frame(index)
    else:
        preds = pd.read_csv(path + file_name, header = [0, 1], index_col = 0)

    preds = preds.dropna().iloc[:, :-1]
    preds.columns = pd.MultiIndex.from_frame(pd.DataFrame(index=preds.columns).reset_index().astype(float))
    return preds


def compute_theoretical_gap(x, betas, cluster, times_eval):
    """
    Ground truth gap between the competing-risks CIF and the probability each
    cause would have if it faced no competition, overall and split by group.
    """
    gap = compute_cif_marginal_gap(x, betas, cluster, times_eval)
    cs = gap['diff']
    return pd.DataFrame({'Overall': cs.mean(axis = 0),
                          'Group': cs[cluster == 0].mean(axis = 0) - cs[cluster == 1].mean(axis = 0)})


def select_at_horizons(preds, times_eval):
    """Nearest column to each evaluation horizon, per cause (keeps the (cause, time) MultiIndex)."""
    nearest = preds[1].iloc[:, np.searchsorted(preds[1].columns.values, times_eval) - 1]
    return nearest

In [ ]:
# Open file and compute performance
for file_name in sorted(os.listdir(path)):
    if 'generate=' not in file_name or '.csv' not in file_name:
        continue

    if f'cause={CAUSE}' not in file_name or f'dim={DIM}' not in file_name:
        continue

    random_seed, model_code = parse_seed_and_model(file_name)
    if model_code not in dict_name:
        continue
    model = dict_name[model_code]
    print("Opening :", file_name, ' - ', model, ' - ', random_seed)

    if model not in results:
        results[model], predictions[model] = {}, {}

    preds = load_predictions(path, file_name)
    times = preds.columns.get_level_values(1).unique()

    # Generate associated ground truth - because sorted groups files by seed, does not require to redo
    if random_seed not in theoretical:
        x, t, e, betas, outcomes = generate(random_seed, causes = list(range(1, CAUSE + 1)), dim = DIM)
        times_eval = np.quantile(t[e == 1], horizons)
        cifs = compute_cif(x, betas, outcomes.cluster, times)

        selection = preds.index
        out = outcomes.loc[selection]
        theoretical[random_seed] = compute_theoretical_gap(x.loc[selection], betas, out.cluster, times_eval)

    # Evaluate
    results[model][random_seed] = evaluate(preds, cifs, e, t, times, times_eval, outcomes.cluster)

    # Keep only times_eval in predictions horizons, per cause
    predictions[model][random_seed] = pd.concat([select_at_horizons(preds, times_eval), outcomes.cluster], axis = 1)

results = pd.concat({model: pd.concat(results[model], names = ['Seed']) for model in results})
results.index.set_names('Model', level = 0, inplace = True)

In [ ]:
# Matched between competing and non competing
matched = {
    'NeuralFG': 'NeuralFG Non Competing',
    'DeSurv': 'DeSurv Non Competing',
    'DeepHit': 'DeepHit Non Competing',
    "Fine-Gray": "Cox"
}

In [ ]:
# Compute empirical error

def compute_empirical_gap(pred_comp, pred_noncomp, index):
    """Empirical gap (non-competing minus competing) at each (cause, time) column, overall and by group."""
    membership = pred_comp.cluster == 0
    overall, group = [], []
    for te in pred_comp.columns[:-1]:
        ratio = (1 - pred_noncomp[te]) - (1 - pred_comp[te])
        overall.append(np.nanmean(ratio))
        group.append(np.nanmean(ratio[membership]) - np.nanmean(ratio[~membership]))
    return pd.DataFrame({'Overall': overall, 'Group': group}, index = index)


empirical = {}
for comp, noncomp in matched.items():
    if comp not in predictions or noncomp not in predictions:
        continue
    common_seeds = set(predictions[comp]) & set(predictions[noncomp])
    empirical[comp] = {seed: compute_empirical_gap(predictions[comp][seed], predictions[noncomp][seed], theoretical[seed].index)
                        for seed in common_seeds}

In [ ]:
# Verify that all folders have same number
results[(results.index.get_level_values('Risk') == 1) & (results.index.get_level_values('Metric') == "SE")].groupby('Model').count()

### Table performance

In [ ]:
# Compute average performance across fold and models
table = results.groupby(['Model', 'Risk', 'Metric']).apply(lambda x:  pd.Series(["{:.3f} ({:.3f})".format(mean, std) for mean, std in zip(x.mean(), x.std())], index = x.columns))
table = table.unstack(level=-1).stack(level=0).unstack(level=-1).loc[:, ['SE']]
table = table.reorder_levels(['Risk', 'Model']).sort_index(level = 0, sort_remaining = False)

table 

In [ ]:
print(table.loc[:,table.columns.get_level_values(1) != 'Overall'].to_latex())

### Display bias error given theoretical error

In [ ]:
from sklearn.metrics import mean_squared_error

In [ ]:
fig, axes = plt.subplots(1, len(horizons), sharey = True, figsize = (10, 5))
slope_dis = {name: {} for name in matched.keys()}
for ite, te in enumerate(horizons):
    true = np.array([theoretical[s].iloc[ite].Overall for s in theoretical])
    for model in empirical:
        estimate = np.array([empirical[model][s].iloc[ite].Overall for s in theoretical])  
        e, t = estimate[~np.isnan(estimate)], true[~np.isnan(estimate)]

        boostrap = []
        np.random.seed(0)   
        for _ in range(100):
            choice = np.random.choice(len(e), size = len(e), replace=True)
            boostrap.append(np.sqrt(mean_squared_error(t[choice], e[choice])))
        slope_dis[model][te] = '{:.3f} ({:.3f})'.format(np.mean(boostrap), np.std(boostrap))

        sns.regplot(x=t, y=e, label = model, 
                    scatter_kws = {'alpha': 0.35, 's': 75}, ci = 0, ax = axes[ite])
    true = np.sort(true)
    axes[ite].plot(true, true, ls = "--", c = 'k', label = "Theory = Experiment")

axes[0].set_title(r"Time = $q_{0.5}$")
axes[1].set_title(r"Time = $q_{1}$")

axes[0].set_ylabel('Empirical')
axes[1].set_xlabel('Theoretical Calibration Error')

plt.legend(loc='center left', bbox_to_anchor=(1, 0.5))

In [ ]:
fig, axes = plt.subplots(1, len(horizons), sharey = True, figsize = (10, 5))
slope_diff = {name: {} for name in matched.keys()}
for ite, te in enumerate(horizons):
    true =  np.array([theoretical[s].iloc[ite].Group for s in theoretical])
    for model in empirical:
        estimate = np.array([empirical[model][s].iloc[ite].Group for s in theoretical])
        e, t = estimate[~np.isnan(estimate)], true[~np.isnan(estimate)]

        boostrap = []
        np.random.seed(0)
        for _ in range(100):
            choice = np.random.choice(len(e), size = len(e), replace=True)
            boostrap.append(np.sqrt(mean_squared_error(t[choice], e[choice])))
        slope_diff[model][te] = '{:.3f} ({:.3f})'.format(np.mean(boostrap), np.std(boostrap))

        sns.regplot(x=true, y=estimate, label = model, 
                    scatter_kws = {'alpha': 0.35, 's': 75}, ci = 0, ax = axes[ite])
    true = np.sort(true)
    axes[ite].plot(true, true, ls = "--", c = 'k', label = "Theory = Experiment")

axes[0].set_title(r"Time = $q_{0.5}$")
axes[1].set_title(r"Time = $q_{1}$")

axes[0].set_ylabel('Empirical')
axes[1].set_xlabel('Theoretical Calibration Gap')

plt.legend(loc='center left', bbox_to_anchor=(1, 0.5))